# RL Creativity Training (English) — Explained Version

This notebook reproduces the RL pipeline but adds explicit explanations of **how each part works** and **why each tool/model is used**.

## 1) Setup

`transformers` loads LLMs/tokenizers, `peft` enables LoRA training, and `torch` handles optimization.

In [ ]:
# !pip install -q transformers peft accelerate bitsandbytes sentencepiece matplotlib pandas numpy torch

In [ ]:
import os, re, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import List, Dict, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, set_seed
from peft import LoraConfig, get_peft_model

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 2) Config

This section defines model choices, RL hyperparameters, reward weights, and KL strength.

In [ ]:
@dataclass
class CFG:
    policy_model_name: str = "distilgpt2"      # trainable policy
    ref_model_name: str = "distilgpt2"         # frozen reference for KL
    ppl_model_name: str = "gpt2"               # frozen PPL evaluator
    judge_model_name: str = "google/flan-t5-large"  # stronger writer judge

    max_new_tokens: int = 64
    temperature: float = 1.0
    top_p: float = 0.95

    lr: float = 2e-5
    train_steps: int = 120
    batch_size: int = 4

    kl_beta: float = 0.05
    judge_weight: float = 0.30
    ppl_weight: float = 0.20
    ngram_weight: float = 0.25
    lexical_weight: float = 0.25

    eval_every: int = 10

cfg = CFG()
print(cfg)


## 3) Prompts

Train prompts drive policy learning; heldout prompts are used for before/after evaluation.

In [ ]:
train_prompts = [
    "Write a short story opening about a city that floats over the ocean.",
    "Invent a new festival tradition and describe how it began.",
    "Imagine a future classroom where emotions are visible as colors.",
    "Describe an unexpected friendship between a robot and a gardener.",
    "Write a paragraph where rain has memory.",
    "Create a travel brochure for a moon made of glass.",
    "Describe an ordinary object that secretly changes history.",
    "Write a scene in which silence is sold in a market.",
    "Explain a new sport played with shadows.",
    "Tell a tale about a library that rewrites readers.",
]

heldout_prompts = [
    "Write a creative paragraph about a museum of forgotten dreams.",
    "Describe a dialogue between winter and a streetlamp.",
    "Imagine a village where names are traded.",
    "Write an opening for a story about edible constellations.",
]


## 4) Load models

- Policy: trainable with LoRA.
- Reference: frozen, defines the original behavior for KL penalty.
- PPL model: frozen fluency proxy.
- Judge model: stronger model for writing-quality reward.

In [ ]:
policy_tok = AutoTokenizer.from_pretrained(cfg.policy_model_name)
if policy_tok.pad_token is None:
    policy_tok.pad_token = policy_tok.eos_token

base_policy = AutoModelForCausalLM.from_pretrained(cfg.policy_model_name)
base_policy.config.pad_token_id = policy_tok.pad_token_id

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
policy_model = get_peft_model(base_policy, lora_cfg).to(device)
policy_model.print_trainable_parameters()

ref_model = AutoModelForCausalLM.from_pretrained(cfg.ref_model_name).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

ppl_tok = AutoTokenizer.from_pretrained(cfg.ppl_model_name)
if ppl_tok.pad_token is None:
    ppl_tok.pad_token = ppl_tok.eos_token
ppl_model = AutoModelForCausalLM.from_pretrained(cfg.ppl_model_name).to(device)
ppl_model.eval()
for p in ppl_model.parameters():
    p.requires_grad = False

judge_tok = AutoTokenizer.from_pretrained(cfg.judge_model_name)
judge_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.judge_model_name).to(device)
judge_model.eval()
for p in judge_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=cfg.lr)
print("Models loaded")


## 5) Reward proxies

This is the core: we compute 4 signals and combine them into one RL reward.

In [ ]:
word_re = re.compile(r"[A-Za-z']+")
stopwords = {
    "the","a","an","in","on","at","to","for","of","and","or","but","if","then",
    "is","are","was","were","be","been","being","this","that","it","as","with",
    "by","from","about","into","over","under","after","before","between","through",
}

def tokenize_words(text: str) -> List[str]:
    # Basic word tokenization for proxy features
    return [w.lower() for w in word_re.findall(text)]

def perplexity_proxy(text: str) -> float:
    # Lower perplexity -> higher score
    enc = ppl_tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = ppl_model(**enc, labels=enc["input_ids"])
    ppl = torch.exp(out.loss).item()
    return 1.0 / (1.0 + math.log1p(ppl))

def ngram_proxy(text: str, n_values=(2,3,4)) -> float:
    # Unique n-gram ratio encourages elaboration and less repetition
    toks = tokenize_words(text)
    if len(toks) < 4:
        return 0.0
    vals = []
    for n in n_values:
        grams = [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]
        if grams:
            vals.append(len(set(grams)) / len(grams))
    return float(np.mean(vals)) if vals else 0.0

def lexical_proxy(text: str) -> float:
    # Lexical richness proxy (diversity + content density + word complexity)
    toks = tokenize_words(text)
    if not toks:
        return 0.0
    ttr = len(set(toks)) / len(toks)
    content = [w for w in toks if w not in stopwords]
    content_density = len(content) / len(toks)
    avg_word_len = float(np.mean([len(w) for w in toks]))
    len_score = min(avg_word_len / 8.0, 1.0)
    return float(0.45*ttr + 0.35*content_density + 0.20*len_score)

judge_cache: Dict[str, float] = {}

def judge_proxy(text: str) -> float:
    # Stronger LLM judge for writing quality (4th signal)
    if text in judge_cache:
        return judge_cache[text]
    prompt = (
        "Rate the WRITING QUALITY of the text from 1 to 10. "
        "Consider coherence, grammar, and readability. "
        "Return only: SCORE: <number>\n\n"
        f"TEXT:\n{text}"
    )
    enc = judge_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        gen = judge_model.generate(**enc, max_new_tokens=16)
    out = judge_tok.decode(gen[0], skip_special_tokens=True)
    m = re.search(r"(10|[1-9](?:\.\d+)?)", out)
    score = float(m.group(1)) if m else 5.0
    score = max(1.0, min(score, 10.0))
    norm = (score - 1.0) / 9.0
    judge_cache[text] = norm
    return norm

def total_reward(text: str) -> Dict[str, float]:
    # Weighted aggregation of all reward parts
    p = perplexity_proxy(text)
    n = ngram_proxy(text)
    l = lexical_proxy(text)
    j = judge_proxy(text)
    total = cfg.ppl_weight*p + cfg.ngram_weight*n + cfg.lexical_weight*l + cfg.judge_weight*j
    return {"ppl": p, "ngram": n, "lexical": l, "judge": j, "reward": total}


## 6) RL helpers

`sample_response` generates actions; `sequence_logprob` gives the policy score for REINFORCE + KL.

In [ ]:
def sample_response(prompt: str) -> str:
    policy_model.eval()
    enc = policy_tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = policy_model.generate(
            **enc,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=True,
            top_p=cfg.top_p,
            temperature=cfg.temperature,
            pad_token_id=policy_tok.pad_token_id,
        )
    text = policy_tok.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text

def sequence_logprob(model, prompt: str, continuation: str) -> torch.Tensor:
    text = prompt + " " + continuation
    enc = policy_tok(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.set_grad_enabled(model.training):
        logits = model(**enc).logits[:, :-1, :]
    labels = enc["input_ids"][:, 1:]
    logp = F.log_softmax(logits, dim=-1)
    tok_logp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    return tok_logp.mean()


## 7) RL training (REINFORCE + KL)

We maximize reward while penalizing divergence from original behavior using KL term.

In [ ]:
history = {"step": [], "loss": [], "reward": [], "kl": [], "regret": [], "ppl": [], "ngram": [], "lexical": [], "judge": []}
running_baseline = 0.0
best_reward = -1e9
policy_model.train()

for step in range(1, cfg.train_steps + 1):
    optimizer.zero_grad()
    prompts = random.sample(train_prompts, k=min(cfg.batch_size, len(train_prompts)))

    batch_loss = 0.0
    batch_rewards, batch_kls = [], []
    comps = {"ppl": [], "ngram": [], "lexical": [], "judge": []}

    for prompt in prompts:
        continuation = sample_response(prompt)
        scores = total_reward(continuation)
        R = scores["reward"]

        # log pi_theta(y|x)
        logp_policy = sequence_logprob(policy_model, prompt, continuation)

        # log pi_ref(y|x) for KL constraint
        with torch.no_grad():
            logp_ref = sequence_logprob(ref_model, prompt, continuation)

        kl_est = torch.clamp(logp_policy - logp_ref, min=-5, max=5)

        advantage = R - running_baseline
        loss = -(advantage * logp_policy) + cfg.kl_beta * kl_est
        batch_loss = batch_loss + loss

        batch_rewards.append(R)
        batch_kls.append(kl_est.detach().item())
        for k in comps:
            comps[k].append(scores[k])

    batch_loss = batch_loss / len(prompts)
    batch_loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    optimizer.step()

    avg_reward = float(np.mean(batch_rewards))
    avg_kl = float(np.mean(batch_kls))

    running_baseline = 0.9 * running_baseline + 0.1 * avg_reward
    best_reward = max(best_reward, avg_reward)
    regret = best_reward - avg_reward

    history["step"].append(step)
    history["loss"].append(float(batch_loss.detach().item()))
    history["reward"].append(avg_reward)
    history["kl"].append(avg_kl)
    history["regret"].append(regret)
    for k in comps:
        history[k].append(float(np.mean(comps[k])))

    if step % cfg.eval_every == 0:
        print(f"step={step:4d} loss={history['loss'][-1]:.4f} reward={avg_reward:.4f} kl={avg_kl:.4f} regret={regret:.4f}")

print("Training finished")


## 8) Plot loss + regret

In [ ]:
df_hist = pd.DataFrame(history)
fig, axs = plt.subplots(1, 3, figsize=(16,4))

axs[0].plot(df_hist["step"], df_hist["loss"])
axs[0].set_title("RL Loss")

axs[1].plot(df_hist["step"], df_hist["reward"], label="reward")
axs[1].plot(df_hist["step"], df_hist["kl"], label="kl")
axs[1].set_title("Reward and KL")
axs[1].legend()

axs[2].plot(df_hist["step"], df_hist["regret"], color="crimson")
axs[2].set_title("Regret")

plt.tight_layout()
plt.show()


## 9) Before/After text samples

Compares untouched base model vs. RL-updated policy on heldout prompts.

In [ ]:
orig_model = AutoModelForCausalLM.from_pretrained(cfg.policy_model_name).to(device)
orig_model.eval()

def generate_with(model, prompt: str, max_new_tokens=80):
    enc = policy_tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.95,
            temperature=1.0,
            pad_token_id=policy_tok.pad_token_id,
        )
    text = policy_tok.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip() if text.startswith(prompt) else text

rows = []
for p in heldout_prompts:
    rows.append({"prompt": p, "before": generate_with(orig_model, p), "after": generate_with(policy_model, p)})

cmp_df = pd.DataFrame(rows)
cmp_df


## 10) Strong model evaluation (fluency + creativity)

Uses the stronger judge model to score before/after outputs and compute deltas.

In [ ]:
def judge_fluency_creativity(text: str) -> Tuple[float, float]:
    prompt = (
        "Rate the following text with two scores from 1 to 10.\n"
        "1) Fluency (grammar/coherence/readability)\n"
        "2) Creativity (novelty/elaboration/original expression)\n"
        "Return exactly: FLUENCY=<num>; CREATIVITY=<num>\n\n"
        f"TEXT:\n{text}"
    )
    enc = judge_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        gen = judge_model.generate(**enc, max_new_tokens=24)
    out = judge_tok.decode(gen[0], skip_special_tokens=True)

    f = re.search(r"FLUENCY\s*=\s*(10|[1-9](?:\.\d+)?)", out)
    c = re.search(r"CREATIVITY\s*=\s*(10|[1-9](?:\.\d+)?)", out)
    flu = float(f.group(1)) if f else 5.0
    cre = float(c.group(1)) if c else 5.0
    return max(1, min(flu, 10)), max(1, min(cre, 10))

scores = []
for row in rows:
    b_f, b_c = judge_fluency_creativity(row["before"])
    a_f, a_c = judge_fluency_creativity(row["after"])
    scores.append({
        "prompt": row["prompt"],
        "before_fluency": b_f,
        "after_fluency": a_f,
        "before_creativity": b_c,
        "after_creativity": a_c,
        "fluency_delta": a_f - b_f,
        "creativity_delta": a_c - b_c,
    })
score_df = pd.DataFrame(scores)
score_df


In [ ]:
print("Average deltas")
print("Fluency Δ:", score_df["fluency_delta"].mean())
print("Creativity Δ:", score_df["creativity_delta"].mean())


## 11) Save artifacts

Saves curves and evaluation tables for your final presentation/report.

In [ ]:
out_dir = "artifacts_rl_creativity_explained"
os.makedirs(out_dir, exist_ok=True)

pd.DataFrame(history).to_csv(f"{out_dir}/train_history.csv", index=False)
cmp_df.to_csv(f"{out_dir}/before_after_generations.csv", index=False)
score_df.to_csv(f"{out_dir}/judge_scores.csv", index=False)

print("Saved in", out_dir)


## 12) GPU scaling notes (RTX 8000)

Start with `distilgpt2`, then scale to `gpt2-medium` or `gpt2-large`; keep LoRA for memory/time efficiency.